# 基于 MindSpore 的 BERT 模型 SWAG 多选阅读理解任务

## 案例介绍

**SWAG** (Situations With Adversarial Generations) 是一个大规模的对抗性数据集，用于基于常识的自然语言推理 (NLI)。给定一个部分描述的事件作为上下文，任务是从四个选项中选择最合理的结尾。

本案例基于 **MindSpore** 框架和 **MindSpore NLP** 套件，使用 **BERT** (Bidirectional Encoder Representations from Transformers) 预训练模型在 SWAG 数据集上进行微调 (Fine-tune)，实现多项选择任务的自动推理。

**核心流程：**
1.  **环境准备**：配置 MindSpore 运行环境及 HF-Mirror 镜像加速。
2.  **数据处理**：加载 SWAG 数据集，进行 Tokenization、Flatten 处理及动态 Padding。
3.  **模型构建**：加载 BERT 预训练权重，构建多选分类网络。
4.  **模型训练**：定义损失函数与优化器，执行微调。
5.  **模型推理**：加载微调后的模型，演示端到端推理。


## 环境准备

本案例运行环境要求如下：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------ |
| 3.9+   | >= 2.7.0    | >= 0.5.1  |

首先导入必要的依赖库，并设置环境变量以使用 HF-Mirror 国内镜像加速下载。


In [ ]:
import os
import mindspore as ms
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

# ----------------------------
# Environment (HF-Mirror)
# ----------------------------
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ----------------------------
# Import MindSpore NLP transformers
# ----------------------------
import mindnlp  # noqa: F401

from mindnlp.transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
)

print(f">>> MindSpore Version: {ms.__version__}")

#### **定义辅助函数**

为了确保代码的健壮性以及在不同硬件（Ascend/GPU/CPU）上的兼容性，我们定义以下工具函数：
- `set_ms_context`: 设置 MindSpore 运行模式（PYNATIVE）。
- `to_numpy`: 鲁棒的 Tensor 转 Numpy 函数，兼容 MindSpore Tensor 和 PyTorch Tensor。
- `move_inputs_to_device`: 确保推理时输入数据与模型在同一设备上。


In [ ]:
# ----------------------------
# MindSpore context
# ----------------------------
def set_ms_context():
    ms.set_seed(42)
    ms.set_context(mode=ms.PYNATIVE_MODE)
    try:
        ms.set_device("Ascend", 0)
    except AttributeError:
        ms.set_context(device_target="Ascend", device_id=0)

def to_numpy(x) -> np.ndarray:
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, (list, tuple)):
        return np.asarray(x)

    # MindSpore Tensor
    if hasattr(x, "asnumpy"):
        try:
            return x.asnumpy()
        except Exception:
            pass

    # (mind)torch Tensor
    if hasattr(x, "detach"):
        try:
            x = x.detach()
        except Exception:
            pass
    if hasattr(x, "cpu"):
        try:
            x = x.cpu()
        except Exception:
            pass
    if hasattr(x, "numpy"):
        try:
            return x.numpy()
        except Exception:
            pass

    return np.asarray(x)

def get_model_device(model):
    """从参数上取 device（适配 mindtorch/torch 风格模型）。"""
    try:
        for p in model.parameters():
            return p.device
    except Exception:
        return None
    return None

def move_inputs_to_device(inputs: Dict[str, Any], device):
    """把 Batch inputs 全部移动到同一 device（仅对有 .to 的张量生效）。"""
    if device is None:
        return inputs
    out = {}
    for k, v in inputs.items():
        if hasattr(v, "to"):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out

# 初始化上下文
set_ms_context()
print(">>> Context set to PYNATIVE | Ascend:0")

## 数据加载与预处理

我们使用 HuggingFace `datasets` 库加载 SWAG 数据集。
为了演示效率，我们从原始训练集和验证集中截取部分数据进行训练。

- **model_checkpoint**: 使用 `google-bert/bert-base-uncased`。


In [ ]:
# 配置
model_checkpoint = "google-bert/bert-base-uncased"
output_dir = "./my_awesome_swag_model_ms"

# 样本量
max_train_samples = 2000
max_eval_samples = 1000

print(f">>> Model: {model_checkpoint}")

# 1. Dataset
from datasets import load_dataset
raw = load_dataset("swag", "regular")

# 截取子集
raw["train"] = raw["train"].select(range(min(max_train_samples, len(raw["train"]))))
raw["validation"] = raw["validation"].select(range(min(max_eval_samples, len(raw["validation"]))))

print(f">>> Data: Train={len(raw['train'])}, Valid={len(raw['validation'])}")

#### **数据预处理 (Tokenization)**

多选任务的数据预处理稍显特殊。我们需要将 **Context (sent1)** 与 **Header (sent2)** 结合，分别与 4 个 **Ending** 选项拼接，形成 4 个独立的输入序列。

1. **Flatten**: 将 `(Batch, 4)` 的结构展平为 `(Batch * 4)` 进行 Tokenize。
2. **Tokenize**: 使用 BERT Tokenizer 进行编码。
3. **Un-flatten**: 将编码后的结果重新 reshape 回 `(Batch, 4, Seq_Len)`。


In [ ]:
# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# 3. Preprocess
ending_names = ["ending0", "ending1", "ending2", "ending3"]

def preprocess_function(examples):
    first_sentences = [[c] * 4 for c in examples["sent1"]]
    question_headers = examples["sent2"]
    second_sentences = [
        [f"{h} {examples[end][i]}" for end in ending_names]
        for i, h in enumerate(question_headers)
    ]

    # Flatten
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(first_sentences, second_sentences, truncation=True)

    # Un-flatten
    result = {k: [v[i:i + 4] for i in range(0, len(v), 4)] for k, v in tokenized.items()}
    result["labels"] = examples["label"]
    return result

# 执行 Map 操作
encoded = raw.map(preprocess_function, batched=True, remove_columns=raw["train"].column_names)
print("Data preprocessing completed.")

#### **定义 DataCollator**

定义数据整理器，负责在 Batch 层面进行 **Dynamic Padding**，并将数据转换为 MindSpore Tensor。


In [ ]:
# ----------------------------
# DataCollator
# ----------------------------
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    padding: Union[bool, str] = "longest"
    pad_to_multiple_of: Optional[int] = None
    label_dtype: ms.dtype = ms.int32

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        label_key = "labels" if "labels" in features[0] else ("label" if "label" in features[0] else None)
        labels = [f.pop(label_key) for f in features] if label_key else None

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = []
        for feat in features:
            for i in range(num_choices):
                flattened.append({k: v[i] for k, v in feat.items()})

        # 训练阶段：优先返回 MindSpore 张量
        try:
            batch = self.tokenizer.pad(
                flattened,
                padding=self.padding,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors="ms",
            )
            out = {k: v.reshape((batch_size, num_choices, -1)) for k, v in batch.items()}
            if labels is not None:
                out["labels"] = ms.Tensor(np.asarray(labels, dtype=np.int32), dtype=self.label_dtype)
            return out
        except Exception:
            # Fallback
            batch_np = self.tokenizer.pad(
                flattened,
                padding=self.padding,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors="np",
            )
            out = {}
            for k, v in batch_np.items():
                arr = np.asarray(v).reshape((batch_size, num_choices, -1))
                if k in ("input_ids", "attention_mask", "token_type_ids"):
                    arr = arr.astype(np.int32, copy=False)
                out[k] = ms.Tensor(arr)
            if labels is not None:
                out["labels"] = ms.Tensor(np.asarray(labels, dtype=np.int32), dtype=self.label_dtype)
            return out

## 模型构建

使用 `AutoModelForMultipleChoice` 加载预训练模型。


In [ ]:
# 4. Model
model = AutoModelForMultipleChoice.from_pretrained(model_checkpoint)
print("Model loaded.")

## 模型训练

配置 `TrainingArguments` 并初始化 `Trainer`。
我们定义 `compute_metrics` 函数来计算准确率 (Accuracy)。


In [ ]:
# 5. Metrics
def compute_metrics(eval_predictions):
    if hasattr(eval_predictions, "predictions"):
        logits = eval_predictions.predictions
        labels = eval_predictions.label_ids
    else:
        logits, labels = eval_predictions
    preds = np.argmax(to_numpy(logits), axis=1)
    labels = to_numpy(labels)
    return {"accuracy": float((preds == labels).mean())}

# 6. TrainingArguments
train_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    push_to_hub=False,
    remove_unused_columns=False,
    report_to=[],
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer),
    compute_metrics=compute_metrics,
)

#### **执行训练与保存**

调用 `trainer.train()` 开始训练，训练结束后保存模型和 Tokenizer。


In [ ]:
# 8. Run
print("\n>>> Starting training...")
trainer.train()

print("\n>>> Starting evaluation...")
metrics = trainer.evaluate()
print(f">>> Eval metrics: {metrics}")

# 9. Save
os.makedirs(output_dir, exist_ok=True)
trainer.save_model(output_dir)
try:
    tokenizer.save_pretrained(output_dir)
except Exception:
    pass
print(f">>> Model saved to: {output_dir}")

## 模型推理

为了验证模型效果，我们进行一次端到端的推理演示。

**注意（关键修复）：**
在 MindSpore NLP 环境下，为了确保推理的稳定性和跨后端兼容性，我们采取以下策略：
1.  **`return_tensors="pt"`**: 使用 PyTorch 兼容的 Tensor 格式（MindSpore NLP 会自动代理到 MindTorch）。
2.  **`move_inputs_to_device`**: 显式将输入数据移动到模型参数所在的设备，避免 "All tensor arguments must be on the same device" 错误。


In [ ]:
# 10. Inference Demo (FIXED)
print("\n>>> Running Inference Demo...")

# 推理阶段需要 torch/no_grad；在 MindSpore NLP 环境下 torch 会被代理到 mindtorch
import torch

model.eval()
device = get_model_device(model)
print(f">>> Inference model device: {device}")

# 1. 准备单条样本
sample = raw["validation"][0]
context = sample["sent1"]
header = sample["sent2"]
choices = [sample[e] for e in ending_names]

first = [context] * 4
second = [f"{header} {c}" for c in choices]

# 2. Tokenize
tok = tokenizer(first, second, truncation=True, padding=True, return_tensors="pt")
inputs = {k: v.reshape((1, 4, -1)) for k, v in tok.items()}
inputs = move_inputs_to_device(inputs, device)

# 3. 执行前向计算 (No Grad)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
pred = int(np.argmax(to_numpy(logits), axis=1)[0])

# 4. 打印结果
print("-" * 50)
print(f"Context: {context}")
print(f"Header : {header}")
for i, c in enumerate(choices):
    mark = "[x]" if i == pred else "[ ]"
    print(f"  {mark} {c}")
print("-" * 50)

gold = int(sample["label"])
if pred == gold:
    print(f"Result: CORRECT (Pred: {pred}, Gold: {gold})")
else:
    print(f"Result: INCORRECT (Pred: {pred}, Gold: {gold})")